# Práctica Guiada: Introducción al Procesamiento del Lenguaje Natural (NLP) con Python

En este cuaderno aprenderás los fundamentos esenciales del Procesamiento del Lenguaje Natural (NLP). Cubriremos las etapas estándar del preprocesamiento de texto, la representación de texto mediante vectores (Bag of Words y TF-IDF) y el entrenamiento de un modelo de clasificación de texto.

---

## Objetivos de la Práctica
1. **Limpieza y Preprocesamiento:** Normalización, tokenización, eliminación de *stop words* y lematización.
2. **Representación de Texto:** Convertir texto libre en vectores numéricos usando `CountVectorizer` y `TfidfVectorizer`.
3. **Clasificación:** Entrenar un modelo para clasificar opiniones/reseñas como **Positivas** o **Negativas**.
4. **Evaluación:** Analizar la precisión de nuestro modelo.

---

## 1. Importación de Librerías y Configuración
Comenzamos cargando las librerías necesarias. Usaremos **NLTK** para el procesamiento de texto y **scikit-learn** para la vectorización y el modelo de machine learning.

In [1]:
import nltk
import re
import pandas as pd
import numpy as np

# Descargar recursos necesarios de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("✅ Librerías cargadas correctamente.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...


✅ Librerías cargadas correctamente.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


---

## 2. Creación del Dataset de Ejemplo
Para esta práctica guiada, crearemos un conjunto de datos ficticio con reseñas sobre productos/servicios.

In [2]:
# Dataset de reseñas de ejemplo
data = {
    'texto': [
        "El producto es excelente, me encantó la calidad y llegó muy rápido.",
        "Pésimo servicio, la atención fue terrible y el producto llegó roto.",
        "Muy buena experiencia de compra, lo recomiendo totalmente.",
        "Una basura de producto, no funciona y perdí mi dinero.",
        "Me gustó bastante, cumple con lo prometido aunque el envío tardó.",
        "Horrible, no lo compren jamás, es una estafa total.",
        "La mejor compra que he hecho este año, fantástico rendimiento.",
        "Muy mal empaquetado y el soporte técnico no me ayudó para nada."
    ],
    'etiqueta': [1, 0, 1, 0, 1, 0, 1, 0]  # 1: Positivo, 0: Negativo
}

df = pd.DataFrame(data)
df

,texto,etiqueta
0,"El producto es excelente, me encantó la calida...",1
1,"Pésimo servicio, la atención fue terrible y el...",0
2,"Muy buena experiencia de compra, lo recomiendo...",1
3,"Una basura de producto, no funciona y perdí mi...",0
4,"Me gustó bastante, cumple con lo prometido aun...",1
5,"Horrible, no lo compren jamás, es una estafa t...",0
6,"La mejor compra que he hecho este año, fantást...",1
7,Muy mal empaquetado y el soporte técnico no me...,0


---

## 3. Preprocesamiento de Texto

El texto sin estructurar contiene ruido (mayúsculas, puntuación, palabras vacías) que dificulta el análisis. Aplicaremos una función para limpiar y estandarizar el texto:

1. **Convertir a minúsculas:** Elimina diferencias entre palabras con distintas mayúsculas.
2. **Eliminar caracteres especiales y números:** Conservamos solo letras y espacios.
3. **Tokenización:** Dividir el texto en palabras individuales.
4. **Eliminar Stop Words:** Filtrar palabras comunes sin carga semántica útil (ej. *el*, *la*, *de*).
5. **Lematización:** Reducir las palabras a su forma raíz o lema.

In [3]:
# Instanciar el lematizador y cargar stop words en español
lemmatizer = WordNetLemmatizer()
stop_words_es = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    # 1. Convertir a minúsculas
    texto = texto.lower()
    
    # 2. Eliminar puntuación y caracteres especiales
    texto = re.sub(r'[^a-záéíóúñ\s]', '', texto)
    
    # 3. Tokenización
    tokens = word_tokenize(texto)
    
    # 4. Eliminar Stop Words y 5. Lematizar
    tokens_limpios = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_es]
    
    # Reconstruir el texto limpio
    return " ".join(tokens_limpios)

# Aplicar limpieza al dataset
df['texto_limpio'] = df['texto'].apply(limpiar_texto)
df[['texto', 'texto_limpio']]

,texto,texto_limpio
0,"El producto es excelente, me encantó la calida...",producto excelente encantó calidad llegó rápido
1,"Pésimo servicio, la atención fue terrible y el...",pésimo servicio atención terrible producto lle...
2,"Muy buena experiencia de compra, lo recomiendo...",buena experiencia compra recomiendo totalmente
3,"Una basura de producto, no funciona y perdí mi...",basura producto funciona perdí dinero
4,"Me gustó bastante, cumple con lo prometido aun...",gustó bastante cumple prometido aunque envío t...
5,"Horrible, no lo compren jamás, es una estafa t...",horrible compren jamás estafa total
6,"La mejor compra que he hecho este año, fantást...",mejor compra hecho año fantástico rendimiento
7,Muy mal empaquetado y el soporte técnico no me...,mal empaquetado soporte técnico ayudó


---

## 4. Vectorización (Representación Numérica)

Los algoritmos de Machine Learning trabajan con números, no con texto. Convertiremos el texto procesado en matrices numéricas usando dos enfoques clásicos:

### A. Bag of Words (CountVectorizer)
Cuenta la frecuencia absoluta de aparición de cada palabra en cada documento.

### B. TF-IDF (Term Frequency - Inverse Document Frequency)
Mide la importancia de una palabra en un documento con relación a todo el corpus. Disminuye la importancia de palabras muy frecuentes en general e incrementa las palabras clave específicas.

In [4]:
# 4.1 Bag of Words
vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(df['texto_limpio'])

print("--- Bag of Words ---")
print("Vocabulario:", vectorizer_bow.get_feature_names_out())
print("Forma de la matriz:", X_bow.toarray().shape)
print("\nEjemplo de Matriz BoW (primeras 2 filas):\n", X_bow.toarray()[:2])

--- Bag of Words ---
Vocabulario: ['atención' 'aunque' 'ayudó' 'año' 'bastante' 'basura' 'buena' 'calidad'
 'compra' 'compren' 'cumple' 'dinero' 'empaquetado' 'encantó' 'envío'
 'estafa' 'excelente' 'experiencia' 'fantástico' 'funciona' 'gustó'
 'hecho' 'horrible' 'jamás' 'llegó' 'mal' 'mejor' 'perdí' 'producto'
 'prometido' 'pésimo' 'recomiendo' 'rendimiento' 'roto' 'rápido'
 'servicio' 'soporte' 'tardó' 'terrible' 'total' 'totalmente' 'técnico']
Forma de la matriz: (8, 42)

Ejemplo de Matriz BoW (primeras 2 filas):
 [[0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0
  0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1
  0 0 1 0 0 0]]


In [5]:
# 4.2 TF-IDF
vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(df['texto_limpio'])

print("--- TF-IDF ---")
print("Matriz TF-IDF (primeras 2 filas):\n", np.round(X_tfidf.toarray()[:2], 2))

--- TF-IDF ---
Matriz TF-IDF (primeras 2 filas):
 [[0.   0.   0.   0.   0.   0.   0.   0.44 0.   0.   0.   0.   0.   0.44
  0.   0.   0.44 0.   0.   0.   0.   0.   0.   0.   0.37 0.   0.   0.
  0.32 0.   0.   0.   0.   0.   0.44 0.   0.   0.   0.   0.   0.   0.  ]
 [0.4  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.34 0.   0.   0.
  0.29 0.   0.4  0.   0.   0.4  0.   0.4  0.   0.   0.4  0.   0.   0.  ]]


---

## 5. Entrenamiento del Modelo de Clasificación

Utilizaremos **Naive Bayes (MultinomialNB)**, un algoritmo probabilístico ideal y estándar para clasificación de texto por su rapidez y efectividad con matrices dispersas.

In [24]:
# Dividir datos en conjunto de entrenamiento y prueba
X = df['texto_limpio']
y = df['etiqueta']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Vectorizar usando TF-IDF
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Entrenar modelo Naive Bayes
modelo = MultinomialNB()
modelo.fit(X_train_vec, y_train)

print("✅ Modelo entrenado correctamente.")

✅ Modelo entrenado correctamente.


---

## 6. Evaluación del Modelo

In [25]:
# Predicciones
y_pred = modelo.predict(X_test_vec)

print("Precisión (Accuracy):", accuracy_score(y_test, y_pred))
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred, target_names=['Negativo', 'Positivo']))

Precisión (Accuracy): 0.25

Reporte de Clasificación:
               precision    recall  f1-score   support

    Negativo       0.00      0.00      0.00         3
    Positivo       0.25      1.00      0.40         1

    accuracy                           0.25         4
   macro avg       0.12      0.50      0.20         4
weighted avg       0.06      0.25      0.10         4



C:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Usuario\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

---

## 7. Probando con Nuevas Reseñas
Escribamos frases nuevas que el modelo no ha visto para probar su capacidad de predicción.

In [26]:
nuevas_reseñas = [
    "Me pareció un servicio excelente y la atención impecable",
    "Una pérdida total de dinero, llegó roto y feo"
]

# Preprocesar y vectorizar
nuevas_limpias = [limpiar_texto(r) for r in nuevas_reseñas]
nuevas_vec = vectorizer.transform(nuevas_limpias)

# Predicción
predicciones = modelo.predict(nuevas_vec)

for reseña, pred in zip(nuevas_reseñas, predicciones):
    sentimiento = "Positivo 😊" if pred == 1 else "Negativo 😡"
    print(f"Reseña: '{reseña}' --> Sentimiento: {sentimiento}")

Reseña: 'Me pareció un servicio excelente y la atención impecable' --> Sentimiento: Positivo 😊
Reseña: 'Una pérdida total de dinero, llegó roto y feo' --> Sentimiento: Positivo 😊
